In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import transformers
from transformers import pipeline
import re

In [2]:
data = pd.read_csv('book_reviews_sample.csv')
data.head()

,index,reviewText,rating
0,11494,Clean and funny. A bit busy with all the diffe...,3
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4
2,1463,Good thing that this is a free story. I read i...,1
3,10342,"Action, action, action! Equipment keeps gettin...",4
4,5256,this was hands down the worse book i have ever...,1


In [3]:
model = pipeline("sentiment-analysis",model="finiteautomata/bertweet-base-sentiment-analysis")

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Device set to use cpu


In [4]:
# Add Sentiment Labels
data['label'] = data['reviewText'].apply(lambda review: model(review)[0]['label'])
data.head()

,index,reviewText,rating,label
0,11494,Clean and funny. A bit busy with all the diffe...,3,POS
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,POS
2,1463,Good thing that this is a free story. I read i...,1,NEG
3,10342,"Action, action, action! Equipment keeps gettin...",4,POS
4,5256,this was hands down the worse book i have ever...,1,NEG


In [5]:
# Check whether: high ratings always have positive sentiment
check = data[(data['rating'] >= 3) & (data['label'] == 'NEG')]
print(check)

    index                                         reviewText  rating label
72  11259  The book had a good story, but it did'nt seem ...       3   NEG


In [6]:
# Create review length column (word count)
data['review_length'] = data['reviewText'].apply(
    lambda x: len(x.split())
)

data['length_category'] = pd.cut(
    data['review_length'],
    bins=[0, 10, 20,50, 100],
    labels=['Short', 'Medium', 'Long', 'Very Long']
)

data.head()

,index,reviewText,rating,label,review_length,length_category
0,11494,Clean and funny. A bit busy with all the diffe...,3,POS,20,Medium
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,POS,21,Long
2,1463,Good thing that this is a free story. I read i...,1,NEG,22,Long
3,10342,"Action, action, action! Equipment keeps gettin...",4,POS,15,Medium
4,5256,this was hands down the worse book i have ever...,1,NEG,17,Medium


In [7]:
# Create Custom Sentiment Strength Column
def map_score(analysed_data):
    label = analysed_data[0]['label']
    score = analysed_data[0]['score']
    if (label == 'POS'):
        if (score > 0 and score <= 0.5):
            return 'Mild Positive'
        else:
            return 'Strong Positive'
    elif (label == 'NEG'):
        if (score > 0 and score <= 0.5):
            return 'Mild Negative'
        else:
            return 'Strong Negative'
    else:
        return "Neutral"
data['sentiment_label'] = data['reviewText'].apply(lambda review: map_score(model(review)))
data['sentiment_score'] = data['reviewText'].apply(lambda review: model(review)[0]['score'])
data.head(20)

,index,reviewText,rating,label,review_length,length_category,sentiment_label,sentiment_score
0,11494,Clean and funny. A bit busy with all the diffe...,3,POS,20,Medium,Strong Positive,0.979848
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,POS,21,Long,Strong Positive,0.991661
2,1463,Good thing that this is a free story. I read i...,1,NEG,22,Long,Strong Negative,0.673863
3,10342,"Action, action, action! Equipment keeps gettin...",4,POS,15,Medium,Strong Positive,0.631572
4,5256,this was hands down the worse book i have ever...,1,NEG,17,Medium,Strong Negative,0.983339
5,7277,"Great book packed full with fast cars , crazy ...",4,POS,18,Medium,Strong Positive,0.990678
6,9781,I enjoyed the reader's digest very much. If I ...,4,POS,21,Long,Strong Positive,0.992272
7,4583,This series has been good and I look forward t...,5,POS,21,Long,Strong Positive,0.992887
8,9797,I just could not get into this book.I all ways...,1,NEG,22,Long,Strong Negative,0.973934
9,895,it was good to see where Dan and Elle was. And...,4,POS,21,Long,Strong Positive,0.991083


In [8]:
negative = data[data['label'] == 'NEG'].reset_index(drop=True)
idx = negative['sentiment_score'].idxmax()
data.iloc[idx]['reviewText']

'Alex a sexy hot cop and the PhD candidate. What a match that makes for a great fun and exciting book'

In [12]:
# Find Frequently Used Words in Negative Reviews
negative = data[data['label'] == 'NEG'].reset_index(drop=True)
allwords = negative['reviewText'].apply(lambda x: x.split()).sum()
allwords = pd.Series(allwords).reset_index(drop=True)
allwords.value_counts().idxmax()

'not'